# 01 - Explore

Scaffold only - no pre-written analysis. Load data, then work below.

In [1]:
import sys

sys.path.insert(0, "..")  # repo root, so `from src import ...` resolves regardless of the kernel's cwd

import pandas as pd

from src import clean, confounder, margin, schema
from src import slice as slice_


## Load data

In [2]:
ledger = pd.read_csv("../data/processed/ledger.csv", parse_dates=[schema.TXN_DATE, schema.PAYMENT_DATE])
overhead = pd.read_csv("../data/processed/overhead.csv", parse_dates=[schema.PERIOD_MONTH])


## Compute margin

In [3]:
ledger_with_margin = margin.compute_margin(ledger, overhead, "proportional_by_revenue")
ledger_with_margin[[schema.TXN_ID, schema.REVENUE, margin.ALLOCATED_COST, margin.MARGIN, margin.MARGIN_PCT]].head()


,txn_id,revenue,allocated_cost,margin,margin_pct
0,TXN0002380,427.46,286.814530,140.645470,0.329026
1,TXN0008743,231.03,235.885588,-4.855588,-0.021017
2,TXN0014094,91.06,20.747309,70.312691,0.772158
3,TXN0002859,319.66,223.998641,95.661359,0.299260
4,TXN0017945,468.04,482.900972,-14.860972,-0.031751


## Slice

## Check confounders

Each helper below returns a comparison table, not a verdict. The entities/periods/dimension picked here (top two customers by revenue, the first two months, `origin_id`) are arbitrary choices to exercise the code paths against the synthetic (patternless) dataset - swap in whatever specific comparison you actually want to check once real data replaces it.

In [4]:
from IPython.display import display

ledger_with_margin["_period"] = margin.period_month(ledger_with_margin[schema.TXN_DATE])
periods = sorted(ledger_with_margin["_period"].dropna().unique())
customers_by_revenue = ledger_with_margin.groupby(schema.CUSTOMER_ID)[schema.REVENUE].sum().sort_values(ascending=False)
top_two_customers = customers_by_revenue.index[:2].tolist()

print("stratified_comparison: top two customers, held fixed by origin_id")
display(confounder.stratified_comparison(
    ledger_with_margin,
    fixed_col=schema.ORIGIN_ID,
    compare_col=schema.CUSTOMER_ID,
    group_a=top_two_customers[0],
    group_b=top_two_customers[1],
))

print("period_over_period_excluding: first two months, excluding the top customer")
display(confounder.period_over_period_excluding(
    ledger_with_margin,
    entity_col=schema.CUSTOMER_ID,
    entity_value=customers_by_revenue.index[0],
    period_a=periods[0],
    period_b=periods[1],
    period_col="_period",
))

print("mix_shift_decomposition: same two months, by origin_id")
display(confounder.mix_shift_decomposition(
    ledger_with_margin,
    period_col="_period",
    period_a=periods[0],
    period_b=periods[1],
    group_col=schema.ORIGIN_ID,
))

print("accrual_vs_cash_view: revenue booked by txn_date vs by payment_date")
display(confounder.accrual_vs_cash_view(ledger_with_margin))


stratified_comparison: top two customers, held fixed by origin_id


,origin_id,rate_CUST0005,txn_count_CUST0005,rate_CUST0034,txn_count_CUST0034,gap
0,OVERALL,0.179059,529,0.181470,534,-0.002411
1,LOC001,0.165039,38,0.240142,31,-0.075103
2,LOC002,0.148885,44,0.215327,65,-0.066443
3,LOC003,0.170955,16,0.183351,17,-0.012396
4,LOC004,0.158340,52,0.155269,64,0.003071
5,LOC005,0.223039,72,0.218823,63,0.004217
6,LOC007,0.198551,69,0.173517,56,0.025035
7,LOC008,0.125132,34,0.155408,46,-0.030276
8,LOC009,0.130060,36,0.119058,33,0.011003
9,LOC010,0.210046,91,0.178222,72,0.031824


period_over_period_excluding: first two months, excluding the top customer


,scenario,period_a_value,period_b_value,change,pct_change
0,including_entity,277812.47,300697.59,22885.12,0.082376
1,excluding_entity,271166.95,291223.80,20056.85,0.073965


mix_shift_decomposition: same two months, by origin_id


,origin_id,revenue_a,revenue_b,margin_pct_a,margin_pct_b,rate_effect,mix_effect,total_effect,actual_margin_change,residual
0,LOC001,18622.42,13034.20,0.219176,0.171164,-894.091772,-956.504333,-1850.596105,NaN,NaN
1,LOC002,26430.50,29478.81,0.196968,0.179658,-457.525093,547.652273,90.127180,NaN,NaN
2,LOC003,11383.19,7858.06,0.191534,0.179529,-136.660951,-632.861657,-769.522608,NaN,NaN
3,LOC004,22907.93,29085.04,0.210633,0.262938,1198.197484,1624.196781,2822.394265,NaN,NaN
4,LOC005,39934.84,51504.81,0.230108,0.222096,-319.961494,2569.646695,2249.685201,NaN,NaN
5,LOC007,25603.18,31740.96,0.159459,0.204265,1147.177996,1253.731229,2400.909225,NaN,NaN
6,LOC008,18518.92,21223.00,0.205920,0.210017,75.860261,567.901463,643.761725,NaN,NaN
7,LOC009,18491.88,17724.33,0.229346,0.180441,-904.349727,-138.497349,-1042.847076,NaN,NaN
8,LOC010,44377.58,45369.85,0.199906,0.170572,-1301.791868,169.253496,-1132.538372,NaN,NaN
9,LOC011,21660.67,18478.18,0.198134,0.214080,345.410571,-681.307455,-335.896884,NaN,NaN


accrual_vs_cash_view: revenue booked by txn_date vs by payment_date


,period_month,accrual_total,cash_total,unsettled_txn_count
0,2024-01-01,277812.47,69116.49,63
1,2024-02-01,300697.59,135825.88,78
2,2024-03-01,312873.76,240742.35,81
3,2024-04-01,306487.46,274467.85,76
4,2024-05-01,293131.83,301474.57,67
5,2024-06-01,284990.19,262667.75,68
6,2024-07-01,246323.98,264154.81,72
7,2024-08-01,244860.75,244119.51,95
8,2024-09-01,243417.56,219130.32,79
9,2024-10-01,237368.92,226985.83,85


## Routing (empty-running)

Independent of the margin pipeline above - trips have no cost columns.
Loads the trips + distance-matrix data, detects dead-head legs, and
shows the per-vehicle empty-km picture. `price_empty_km`'s diesel rate
and mileage below are an illustrative example, not a real assumption -
pass your own when you have one.

In [5]:
from src import routing

trips = pd.read_csv("../data/processed/trips.csv", parse_dates=[schema.TRIP_DATE])
distances = pd.read_csv("../data/reference/distances.csv")
trips.shape, distances.shape


((400, 6), (90, 3))

In [6]:
empty_legs = routing.detect_empty_legs(trips, distances)
vehicle_km = routing.summarize_vehicle_km(trips, distances).sort_values("empty_pct", ascending=False)
print(f"{len(empty_legs)} empty legs detected across {trips[schema.VEHICLE_ID].nunique()} vehicles")
vehicle_km


348 empty legs detected across 12 vehicles


,vehicle_id,loaded_km,empty_km,empty_pct,empty_legs,missing_km_legs
8,VEH009,5579.1,5313.4,0.487804,24,0
6,VEH007,6940.6,6521.2,0.484423,31,0
11,VEH012,4822.5,4482.8,0.481747,21,0
3,VEH004,5747.6,5340.4,0.481638,23,0
10,VEH011,8176.8,7318.0,0.472287,39,0
5,VEH006,6739.4,5978.6,0.470090,27,0
7,VEH008,7353.7,6403.5,0.465465,33,0
2,VEH003,6913.8,5867.4,0.459065,32,0
4,VEH005,7385.8,5840.2,0.441570,30,0
0,VEH001,6588.4,4809.8,0.421979,24,0


In [7]:
idle = routing.utilisation(trips)
routing.utilisation_summary(idle)


,vehicle_id,total_idle_days,avg_idle_days,gaps_counted
0,VEH001,324,12.000000,27
1,VEH002,181,4.641026,39
2,VEH003,332,9.764706,34
3,VEH004,96,4.000000,24
4,VEH005,305,9.242424,33
5,VEH006,184,5.750000,32
6,VEH007,124,3.757576,33
7,VEH008,337,9.108108,37
8,VEH009,272,10.074074,27
9,VEH010,183,5.718750,32


In [8]:
# Illustrative example only - substitute your real diesel price (Rs/litre)
# and vehicle mileage (km/litre) before trusting this number.
EXAMPLE_DIESEL_PRICE_PER_LITRE = 95.0
EXAMPLE_KM_PER_LITRE = 4.0

priced = routing.price_empty_km(empty_legs, EXAMPLE_DIESEL_PRICE_PER_LITRE, EXAMPLE_KM_PER_LITRE)
print(f"Example total empty-running cost at Rs{EXAMPLE_DIESEL_PRICE_PER_LITRE}/L, {EXAMPLE_KM_PER_LITRE} km/L:")
print(f"  Rs{priced['empty_cost'].sum():,.2f} across {priced['empty_cost'].notna().sum()} priced legs "
      f"({priced['empty_cost'].isna().sum()} legs excluded - missing distance)")
priced.head()


Example total empty-running cost at Rs95.0/L, 4.0 km/L:
  Rs1,631,653.50 across 348 priced legs (0 legs excluded - missing distance)


,vehicle_id,empty_from,empty_to,date_gap_days,empty_km,km_missing,empty_cost
0,VEH001,LOC009,LOC002,14,131.5,False,3123.125
1,VEH001,LOC001,LOC002,4,172.8,False,4104.000
2,VEH001,LOC008,LOC006,7,319.0,False,7576.250
3,VEH001,LOC002,LOC004,6,68.6,False,1629.250
4,VEH001,LOC005,LOC003,1,276.6,False,6569.250
